<a href="https://colab.research.google.com/github/NazHub1993/ML_Notebooks/blob/main/Heart_Disease_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Heart Disease Detection Dataset in kaggle

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("johnsmith88/heart-disease-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'heart-disease-dataset' dataset.
Path to dataset files: /kaggle/input/heart-disease-dataset


In [3]:
import os
os.listdir(path)

['heart.csv']

In [4]:
#Imports
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder,OrdinalEncoder

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.svm import SVC


In [5]:
csv_file=os.path.join(path,'heart.csv')
df=pd.read_csv(csv_file)
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,52,1,0,125,212,0,1,168,0,1.0,2,2,3,0
1,53,1,0,140,203,1,0,155,1,3.1,0,0,3,0
2,70,1,0,145,174,0,1,125,1,2.6,0,0,3,0
3,61,1,0,148,203,0,1,161,0,0.0,2,1,3,0
4,62,0,0,138,294,1,1,106,0,1.9,1,3,2,0


#We will follow the approach


*   Drop the duplicates
*   Drop unnecessary features

*   Divide into X and Y
*   train-test-split

*   Numerical column- Fill the missing value, cap the outlier,standardscaler
*   Categorical column- Fill the missing value, OneHotEncode, Ordinal encode

*   Create the pipeline
*   Use Grid-Search-CV

*   Select the best-estimator
*   Create the classification matrix











In [6]:
df.shape

(1025, 14)

In [7]:
df.duplicated().sum()

np.int64(723)

In [8]:
print(df.duplicated())

0       False
1       False
2       False
3       False
4       False
        ...  
1020     True
1021     True
1022     True
1023     True
1024     True
Length: 1025, dtype: bool


In [9]:
df[df.duplicated()]

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
15,34,0,1,118,210,0,1,192,0,0.7,2,0,2,1
31,50,0,1,120,244,0,1,162,0,1.1,2,0,2,1
43,46,1,0,120,249,0,0,144,0,0.8,2,0,3,0
55,55,1,0,140,217,0,1,111,1,5.6,0,0,3,0
61,66,0,2,146,278,0,0,152,0,0.0,1,1,2,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1020,59,1,1,140,221,0,1,164,1,0.0,2,0,2,1
1021,60,1,0,125,258,0,0,141,1,2.8,1,1,3,0
1022,47,1,0,110,275,0,0,118,1,1.0,1,1,2,0
1023,50,0,0,110,254,0,0,159,0,0.0,2,0,2,1


#Dropping the duplicates first so that the model doesn't lead to bad performance

In [10]:
df.drop_duplicates().shape

(302, 14)

In [11]:
df.duplicated().sum()

np.int64(723)

In [12]:
df=df.drop_duplicates()

In [13]:
df.shape

(302, 14)

In [14]:
df.columns

Index(['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach',
       'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target'],
      dtype='object')

In [15]:
df.isnull().sum()

,0
age,0
sex,0
cp,0
trestbps,0
chol,0
fbs,0
restecg,0
thalach,0
exang,0
oldpeak,0


In [16]:
df.describe()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
count,302.00000,302.000000,302.000000,302.000000,302.000000,302.000000,302.000000,302.000000,302.000000,302.000000,302.000000,302.000000,302.000000,302.000000
mean,54.42053,0.682119,0.963576,131.602649,246.500000,0.149007,0.526490,149.569536,0.327815,1.043046,1.397351,0.718543,2.314570,0.543046
std,9.04797,0.466426,1.032044,17.563394,51.753489,0.356686,0.526027,22.903527,0.470196,1.161452,0.616274,1.006748,0.613026,0.498970
min,29.00000,0.000000,0.000000,94.000000,126.000000,0.000000,0.000000,71.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,48.00000,0.000000,0.000000,120.000000,211.000000,0.000000,0.000000,133.250000,0.000000,0.000000,1.000000,0.000000,2.000000,0.000000
50%,55.50000,1.000000,1.000000,130.000000,240.500000,0.000000,1.000000,152.500000,0.000000,0.800000,1.000000,0.000000,2.000000,1.000000
75%,61.00000,1.000000,2.000000,140.000000,274.750000,0.000000,1.000000,166.000000,1.000000,1.600000,2.000000,1.000000,3.000000,1.000000
max,77.00000,1.000000,3.000000,200.000000,564.000000,1.000000,2.000000,202.000000,1.000000,6.200000,2.000000,4.000000,3.000000,1.000000


In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 302 entries, 0 to 878
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       302 non-null    int64  
 1   sex       302 non-null    int64  
 2   cp        302 non-null    int64  
 3   trestbps  302 non-null    int64  
 4   chol      302 non-null    int64  
 5   fbs       302 non-null    int64  
 6   restecg   302 non-null    int64  
 7   thalach   302 non-null    int64  
 8   exang     302 non-null    int64  
 9   oldpeak   302 non-null    float64
 10  slope     302 non-null    int64  
 11  ca        302 non-null    int64  
 12  thal      302 non-null    int64  
 13  target    302 non-null    int64  
dtypes: float64(1), int64(13)
memory usage: 35.4 KB


#So there are no missing values

#All the columns are numerical columns

In [18]:
df['age'].unique()

array([52, 53, 70, 61, 62, 58, 55, 46, 54, 71, 43, 34, 51, 50, 60, 67, 45,
       63, 42, 44, 56, 57, 59, 64, 65, 41, 66, 38, 49, 48, 29, 37, 47, 68,
       76, 40, 39, 77, 69, 35, 74])

In [19]:
df['sex'].unique()

array([1, 0])

In [20]:
df['cp'].unique()

array([0, 1, 2, 3])

#cp is a categorical column need to be one hot encoded

In [21]:
df['ca'].unique()

array([2, 0, 1, 3, 4])

# ca,cp,restecg,slope,thal categorical column

In [22]:
df['chol'].unique()

array([212, 203, 174, 294, 248, 318, 289, 249, 286, 149, 341, 210, 298,
       204, 308, 266, 244, 211, 185, 223, 208, 252, 209, 307, 233, 319,
       256, 327, 169, 131, 269, 196, 231, 213, 271, 263, 229, 360, 258,
       330, 342, 226, 228, 278, 230, 283, 241, 175, 188, 217, 193, 245,
       232, 299, 288, 197, 315, 215, 164, 326, 207, 177, 257, 255, 187,
       201, 220, 268, 267, 236, 303, 282, 126, 309, 186, 275, 281, 206,
       335, 218, 254, 295, 417, 260, 240, 302, 192, 225, 325, 235, 274,
       234, 182, 167, 172, 321, 300, 199, 564, 157, 304, 222, 184, 354,
       160, 247, 239, 246, 409, 293, 180, 250, 221, 200, 227, 243, 311,
       261, 242, 205, 306, 219, 353, 198, 394, 183, 237, 224, 265, 313,
       340, 259, 270, 216, 264, 276, 322, 214, 273, 253, 176, 284, 305,
       168, 407, 290, 277, 262, 195, 166, 178, 141])

#chol,age,oldpeak,thalach,trestbps  numeric column

In [23]:
df['exang'].unique()

array([0, 1])

#exang,sex,fbs binary column

In [24]:
df['fbs'].unique()


array([0, 1])

In [25]:
df['oldpeak'].unique()

array([1. , 3.1, 2.6, 0. , 1.9, 4.4, 0.8, 3.2, 1.6, 3. , 0.7, 4.2, 1.5,
       2.2, 1.1, 0.3, 0.4, 0.6, 3.4, 2.8, 1.2, 2.9, 3.6, 1.4, 0.2, 2. ,
       5.6, 0.9, 1.8, 6.2, 4. , 2.5, 0.5, 0.1, 2.1, 2.4, 3.8, 2.3, 1.3,
       3.5])

In [26]:
df['restecg'].unique()

array([1, 0, 2])

In [27]:
df['slope'].unique()

array([2, 0, 1])

In [28]:
df['thal'].unique()

array([3, 2, 1, 0])

In [29]:
df['thalach'].unique()

array([168, 155, 125, 161, 106, 122, 140, 145, 144, 116, 136, 192, 156,
       142, 109, 162, 165, 148, 172, 173, 146, 179, 152, 117, 115, 112,
       163, 147, 182, 105, 150, 151, 169, 166, 178, 132, 160, 123, 139,
       111, 180, 164, 202, 157, 159, 170, 138, 175, 158, 126, 143, 141,
       167,  95, 190, 118, 103, 181, 108, 177, 134, 120, 171, 149, 154,
       153,  88, 174, 114, 195, 133,  96, 124, 131, 185, 194, 128, 127,
       186, 184, 188, 130,  71, 137,  99, 121, 187,  97,  90, 129, 113])

#Dividing the dataset into X and Y

In [30]:
X=df.drop('target',axis=1)
Y=df['target']

In [31]:
Y

,target
0,0
1,0
2,0
3,0
4,0
...,...
723,1
733,1
739,0
843,0


#Train-test-splitting the data

In [32]:
x_train,x_test,y_train,y_test=train_test_split(X,Y,test_size=0.2,random_state=42,stratify=Y)

In [33]:
binary_cols=[col for col in df.columns if set(df[col].dropna().unique()).issubset({0,1})]
numeric_cols=[col for col in df.select_dtypes(include="number").columns if col not in binary_cols]

print("Binary columns:",binary_cols)
print("Numeric columns:",numeric_cols)

Binary columns: ['sex', 'fbs', 'exang', 'target']
Numeric columns: ['age', 'cp', 'trestbps', 'chol', 'restecg', 'thalach', 'oldpeak', 'slope', 'ca', 'thal']


In [34]:
binary_cols.remove('target')

In [35]:
binary_cols

['sex', 'fbs', 'exang']

In [36]:
numeric_cols

['age',
 'cp',
 'trestbps',
 'chol',
 'restecg',
 'thalach',
 'oldpeak',
 'slope',
 'ca',
 'thal']

In [37]:
lo=X[numeric_cols].quantile(0.01)
hi=X[numeric_cols].quantile(0.99)

print(lo)
print(hi)

age          35.00
cp            0.00
trestbps    100.00
chol        149.00
restecg       0.00
thalach      95.01
oldpeak       0.00
slope         0.00
ca            0.00
thal          1.00
Name: 0.01, dtype: float64
age          71.00
cp            3.00
trestbps    180.00
chol        406.87
restecg       1.99
thalach     191.98
oldpeak       4.20
slope         2.00
ca            3.99
thal          3.00
Name: 0.99, dtype: float64


#Using ColumnTransformer and pipeline

In [38]:
from sklearn.preprocessing import FunctionTransformer

In [39]:
from sklearn.preprocessing import FunctionTransformer

ohe_cols     = ["cp", "restecg", "thal", "slope", "ca"]
binary_cols  = ["sex", "fbs", "exang"]
numeric_cols = ["age", "trestbps", "chol", "thalach", "oldpeak"]

lo = x_train[numeric_cols].quantile(0.01)
hi = x_train[numeric_cols].quantile(0.99)

def clip_fixed(X, lo, hi):
    return np.clip(X, lo.values, hi.values)

winsorizer = FunctionTransformer(clip_fixed, kw_args={"lo": lo, "hi": hi})

In [40]:
prep=ColumnTransformer(
    [
        ("binary_transformer","passthrough",binary_cols),
        ("ohe_transformer",OneHotEncoder(handle_unknown="ignore"),ohe_cols),
        ("numeric_transformer",
         Pipeline(
             [
                 ("winsorizer",winsorizer),
                 ("scaler",StandardScaler())

             ]
         )
         ,numeric_cols)
    ]
)


In [41]:
pipe=Pipeline(
    [
        ("prep",prep),
        ("model",SVC())
    ]
)

In [42]:
from sklearn.model_selection import StratifiedKFold
skf=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

#Getting the best hyperparameters using hyperparameter tuning

In [43]:
search=GridSearchCV(
    pipe,
    param_grid={
        "model__kernel":["linear","rbf","poly"],
        "model__C":[0.01, 0.1, 1, 10, 100],
        "model__gamma":[0.01, 0.1, 1, 10, 100]
    },
    cv=skf,
    scoring="accuracy",
    verbose=1
)

In [44]:
search.fit(x_train,y_train)

Fitting 5 folds for each of 75 candidates, totalling 375 fits


GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('prep',
                                        ColumnTransformer(transformers=[('binary_transformer',
                                                                         'passthrough',
                                                                         ['sex',
                                                                          'fbs',
                                                                          'exang']),
                                                                        ('ohe_transformer',
                                                                         OneHotEncoder(handle_unknown='ignore'),
                                                                         ['cp',
                                                                          'restecg',
                                                                          'thal',
                                                                          'slope',
                                                                          'ca']),
                                                                        ('numeric_transformer',
                                                                         Pipeline(steps=[('winsorizer',
                                                                                          Fun...
Name: 0.99, dtype: float64,
                                                                                                                       'lo': age          34.4
trestbps    100.0
chol        144.2
thalach      95.4
oldpeak       0.0
Name: 0.01, dtype: float64})),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         ['age',
                                                                          'trestbps',
                                                                          'chol',
                                                                          'thalach',
                                                                          'oldpeak'])])),
                                       ('model', SVC())]),
             param_grid={'model__C': [0.01, 0.1, 1, 10, 100],
                         'model__gamma': [0.01, 0.1, 1, 10, 100],
                         'model__kernel': ['linear', 'rbf', 'poly']},
             scoring='accuracy', verbose=1)

In [45]:
search.best_score_

np.float64(0.8628401360544217)

In [46]:
# 1. Class balance (decides your scoring metric)
print(Y.value_counts())
print(df['ca'].value_counts())

# 2. Fit the search (this also refits the best model on all of x_train)
search.fit(x_train, y_train)

# 3. Best settings and the three scores
print("Best params:", search.best_params_)
print("CV score:   ", search.best_score_)
print("Train score:", search.score(x_train, y_train))
print("Test score: ", search.score(x_test, y_test))



target
1    164
0    138
Name: count, dtype: int64
ca
0    175
1     65
2     38
3     20
4      4
Name: count, dtype: int64
Fitting 5 folds for each of 75 candidates, totalling 375 fits
Best params: {'model__C': 1, 'model__gamma': 0.01, 'model__kernel': 'rbf'}
CV score:    0.8628401360544217
Train score: 0.8796680497925311
Test score:  0.8360655737704918


In [47]:
best_model = search.best_estimator_
y_pred = best_model.predict(x_test)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.82      0.82      0.82        28
           1       0.85      0.85      0.85        33

    accuracy                           0.84        61
   macro avg       0.83      0.83      0.83        61
weighted avg       0.84      0.84      0.84        61

[[23  5]
 [ 5 28]]


#Write functions that would calculate hinge loss

In [48]:
def hinge_loss(w,b,X,y):
  return np.mean(np.maximum(0,1-y*(X@w + b)))

In [49]:
X=np.array([
    [2, 1],
    [1, 2]
])
y=np.array([[1],[-1]])
w=np.array([[1],[1]])
b=0

ans=hinge_loss(w,b,X,y)
print(ans)

2.0


In [51]:
def rbf_kernel(x1,x2,gamma):
  sqrd_dist=np.sum(x1**2,axis=1)[:,None]+np.sum(x2**2,axis=1)[None,:]-2*x1@x2.T
  return np.exp(-gamma*sqrd_dist)

x1=[[0,0]]
x2=[[3,4]]

gamma=0.01

In [52]:
ans=rbf_kernel(np.array(x1),np.array(x2),gamma)
print(ans)

[[0.77880078]]
